In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
data = pd.read_csv("C:/Users/santo/ml_projects/HousePricePrediction/train.csv")

# Select features and target
features = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea',
            'TotalBsmtSF', 'FullBath', 'YearBuilt']
data = data.dropna(subset=features + ['SalePrice'])

X = data[features]
y = data['SalePrice']

# Train-test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Candidate models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=42)
}

# Train and evaluate models
print("\n📊 Evaluation Results:")
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    print(f"\n🔹 Model: {name}")
    print("MAE:", mean_absolute_error(y_val, preds))
    print("RMSE:", np.sqrt(mean_squared_error(y_val, preds)))
    print("R² Score:", r2_score(y_val, preds))
    trained_models[name] = model

# Hyperparameter tuning for Random Forest
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20]
}
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid,
                           cv=3, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

# Evaluate best random forest
best_preds = best_rf.predict(X_val)
print("\n⭐ Best Random Forest after tuning:")
print("MAE:", mean_absolute_error(y_val, best_preds))
print("RMSE:", np.sqrt(mean_squared_error(y_val, best_preds)))
print("R² Score:", r2_score(y_val, best_preds))

# Save model
with open('trained_model.pkl', 'wb') as f:
    pickle.dump(best_rf, f)

print("\n✅ Model saved as 'trained_model.pkl'")

# Save test data
X_val.to_csv("test.csv", index=False)
y_val.to_csv("house_price_submission.csv", index=False)
print("📄 Test data saved as 'test.csv' and 'house_price_submission.csv'")

# Residual Plot
residuals = y_val - best_preds
plt.figure(figsize=(6,4))
sns.residplot(x=best_preds, y=residuals, lowess=True, color="purple")
plt.title("Residual Plot")
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.grid(True)
plt.tight_layout()
plt.savefig("residual_plot.png")
plt.close()

# Predicted vs Actual Plot
plt.figure(figsize=(6,4))
sns.scatterplot(x=y_val, y=best_preds, color="teal")
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], '--', color='red')
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Price")
plt.grid(True)
plt.tight_layout()
plt.savefig("predicted_vs_actual.png")
plt.close()

print("📈 Plots saved as 'residual_plot.png' and 'predicted_vs_actual.png'")



📊 Evaluation Results:

🔹 Model: Linear Regression
MAE: 25121.61954347192
RMSE: 39652.79061526264
R² Score: 0.7950095261783562

🔹 Model: Ridge Regression
MAE: 25115.797346245272
RMSE: 39651.3300225149
R² Score: 0.7950246273644519

🔹 Model: Random Forest
MAE: 19534.81369169928
RMSE: 30077.850799880194
R² Score: 0.8820549368692863

⭐ Best Random Forest after tuning:
MAE: 19479.31462011994
RMSE: 30247.209090063425
R² Score: 0.880722979306831

✅ Model saved as 'trained_model.pkl'
📄 Test data saved as 'test.csv' and 'house_price_submission.csv'
📈 Plots saved as 'residual_plot.png' and 'predicted_vs_actual.png'
